# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [3]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imnotparama/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())

df["stale_bucket"] = pd.cut(df["days_since_last_update"],
                              bins=[0, 90, 180, 1000],
                              labels=["fresh (<90d)", "aging (90-180d)", "stale (180d+)"])

bucket_table_1 = df.groupby("stale_bucket", observed=True).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
)
print("Signal 1 - Staleness:\n", bucket_table_1)

df["impressions_bucket"] = pd.cut(df["impressions_90d"],
                                    bins=[-1, 100, 500, 1000000],
                                    labels=["low (<100)", "medium (100-500)", "high (500+)"])

bucket_table_2 = df.groupby("impressions_bucket", observed=True).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
)
print("\nSignal 2 - Impressions/volume:\n", bucket_table_2)

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship
Signal 1 - Staleness:
                      n  decline_rate
stale_bucket                        
fresh (<90d)     20655      0.512031
aging (90-180d)   9171      0.611057
stale (180d+)      174      0.471264

Signal 2 - Impressions/volume:
                         n  decline_rate
impressions_bucket                     
low (<100)           8006      0.389208
medium (100-500)     5279      0.604281
high (500+)         16715      0.595633


**Signal 1 — Staleness verdict: MIXED.** Decline rate rises from fresh
(0.512) to aging (0.611), but then drops for truly stale pages (0.471,
n=174 only). Staleness alone doesn't cleanly predict decline — the
relationship isn't monotonic, and very few pages are actually 180+ days
stale.

**Signal 2 — Impressions/volume verdict: MIXED.** Decline rate jumps
from low (0.389) to medium (0.604) impressions, but flattens rather
than continuing to rise for high-impression pages (0.596). Some
relationship exists, but it's not a clean linear signal either.

**My rule:** A page is worth reviewing if it's stale (180+ days since
last update) AND still getting real traffic (100+ impressions in the
last 90 days). Both signals are directionally real but noisy — this is
exactly why the rule is a baseline to beat, not a final answer.

**Reason code this rule outputs:**
- stale_visible_page: stale AND still visible

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
df["baseline_score"] = (
    (df["days_since_last_update"] >= 180).astype(int) *
    (df["impressions_90d"] >= 100).astype(int) *
    df["impressions_90d"]
)
df["reason_code"] = "stale_visible_page"
df["action"] = "review_for_refresh"

queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "baseline_score", "reason_code", "action",
     "days_since_last_update", "impressions_90d", "trend_direction"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows")
queue.head(20)

Wrote 30000 rows


,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,trend_direction
16751,content_cf56e2e2e282,61678,stale_visible_page,review_for_refresh,194,61678,down
16514,content_7368877ea310,59472,stale_visible_page,review_for_refresh,194,59472,down
7021,content_1bfaa38ff26c,25715,stale_visible_page,review_for_refresh,194,25715,down
21268,content_0a91db491d14,13299,stale_visible_page,review_for_refresh,193,13299,down
11489,content_5feee3994adb,7812,stale_visible_page,review_for_refresh,194,7812,down
12045,content_c2d929d83eaa,7558,stale_visible_page,review_for_refresh,193,7558,down
698,content_b16bd7307b39,4590,stale_visible_page,review_for_refresh,194,4590,down
5327,content_fe16a55cd13d,4556,stale_visible_page,review_for_refresh,194,4556,down
26810,content_ecb6215e79fd,4429,stale_visible_page,review_for_refresh,194,4429,down
20837,content_928af3e22c80,1697,stale_visible_page,review_for_refresh,193,1697,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. content_cf56e2e2e282 — action: review_for_refresh. Why: 194 days
   stale, 61,678 impressions/90d, trend down. Would be wrong if: this
   traffic is inflated by a bot/crawler spike rather than real users.
2. content_7368877ea310 — action: review_for_refresh. Why: 194 days
   stale, 59,472 impressions, trend down. Would be wrong if: recent
   consolidation from a sibling page temporarily boosted impressions.
3. content_1bfaa38ff26c — action: review_for_refresh. Why: 194 days
   stale, 25,715 impressions, trend down. Would be wrong if: seasonal
   demand spike, not a real decline signal.
4. content_0a91db491d14 — action: review_for_refresh. Why: 193 days
   stale, 13,299 impressions, trend down. Would be wrong if: this page
   was recently redirected and inherited another page's traffic.
5. content_5feee3994adb — action: review_for_refresh. Why: 194 days
   stale, 7,812 impressions, trend down. Would be wrong if: traffic is
   concentrated in a single spike day, not sustained.
6. content_c2d929d83eaa — action: review_for_refresh. Why: 193 days
   stale, 7,558 impressions, trend down. Would be wrong if: impression
   count includes non-organic (paid/referral) traffic.
7. content_b16bd7307b39 — action: review_for_refresh. Why: 194 days
   stale, 4,590 impressions, trend down. Would be wrong if: this page
   is mid-migration and metrics are temporarily unstable.
8. content_fe16a55cd13d — action: review_for_refresh. Why: 194 days
   stale, 4,556 impressions, trend down. Would be wrong if: the drop is
   explained by a competitor page, not content quality.
9. content_ecb6215e79fd — action: review_for_refresh. Why: 194 days
   stale, 4,429 impressions, trend down. Would be wrong if: page is
   already scheduled for deprecation/removal.
10. content_928af3e22c80 — action: review_for_refresh. Why: 193 days
    stale, 1,697 impressions, trend down. Would be wrong if: low volume
    makes this mostly noise despite crossing the threshold.
11. content_e3f1b093148 — action: review_for_refresh. Why: 183 days
    stale, 1,408 impressions, trend down. Would be wrong if: this is a
    seasonal page naturally quiet right now.
12. content_bdbec75c1148 — action: review_for_refresh. Why: 194 days
    stale, 1,316 impressions, trend stable (not declining). Would be
    wrong if: flagged despite trend being "stable," not "down" —
    weakest pick in the top 20.
13. content_7f116ae1f6f5 — action: review_for_refresh. Why: 301 days
    stale, 954 impressions, trend down. Would be wrong if: content is
    evergreen and doesn't need frequent updates despite staleness.
14. content_77d4d5930e5e — action: review_for_refresh. Why: 194 days
    stale, 828 impressions, trend down. Would be wrong if: low absolute
    volume makes the decline statistically weak.
15. content_72496874f806 — action: review_for_refresh. Why: 301 days
    stale, 821 impressions, trend down. Would be wrong if: same
    low-volume noise concern as above.
16. content_6226ee6adc91 — action: review_for_refresh. Why: 183 days
    stale, 545 impressions, trend down. Would be wrong if: borderline
    on both thresholds — weak signal overall.
17. content_074ba6ead17b — action: review_for_refresh. Why: 183 days
    stale, 533 impressions, trend down. Would be wrong if: same
    borderline concern.
18. content_fd16e3475c29 — action: review_for_refresh. Why: 183 days
    stale, 429 impressions, trend down. Would be wrong if: impressions
    barely clear the 100 threshold with little real signal strength.
19. content_b65fe2792b44 — action: review_for_refresh. Why: 183 days
    stale, 371 impressions, trend UP (not down). Would be wrong if:
    flagged despite trending up — likely the weakest pick in the top 20.
20. content_ba00ffc6318c — action: review_for_refresh. Why: 211 days
    stale, 345 impressions, trend down. Would be wrong if: low volume
    again raises noise concern.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** Row 12 (content_bdbec75c1148) is flagged despite having
trend_direction == "stable", not "down" — my rule doesn't check trend
direction directly, only staleness + impressions, so it can surface
non-declining pages. Row 19 (content_b65fe2792b44) is worse — it's
trending UP, yet still got flagged. This shows a real weakness: my rule
ignores trend_direction entirely, which a smarter rule or model should
fix.

**Leakage check:** No future-window data used. No FlyRank product flags
(health_score, priority_score, action_type) used as inputs — only
observable signals (days_since_last_update, impressions_90d) available
before any decision point. Confirmed no leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.